# EEG_25 — Reliability check della bAcc (gate forti/scarsi)

**Domanda (esperimento #0)**: "soggetto bravo" è un **tratto stabile** o **rumore di stima**?

La bAcc per soggetto (EEG_13b) è stimata da pochi trial → rumorosa. Se la spezzo in due metà
indipendenti, le due bAcc concordano? Se no, non esiste un tratto da predire — e *questo* è il risultato.

**Perché è il gate**: la reliability del target è il **soffitto** della predicibilità. Nessuna feature
EEG può spiegare più varianza di quanta ne sia affidabile nel target. Se r_split-half ≈ 0, qualsiasi
modello (EEG_24, Riemannian, multivariato) è cappato dall'inizio.

**Dato usato**: `models/eeg13b_200e/P*.pt` contiene già `labels` (y_true) e `preds` (y_pred) sul
test set per soggetto. Nessuna inference da rigenerare.

**Metodo**: split-half stratificato ripetuto (StratifiedShuffleSplit 50/50, B ripetizioni) →
bAcc_A vs bAcc_B tra soggetti → Pearson + Spearman + correzione Spearman-Brown. Per cluster C0/C1.


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit

# ---- paths ----
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CKPT_DIR = project_root / 'models' / 'eeg13b_200e'
CLUSTER_JSON = project_root / 'configs' / 'eeg16b_cluster_labels.json'
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

N_REPS = 500          # ripetizioni split-half
SEED0 = 0
rng = np.random.default_rng(SEED0)

def to_np(a):
    if torch.is_tensor(a): return a.detach().cpu().numpy()
    return np.asarray(a)

print('CKPT_DIR:', CKPT_DIR, '| esiste:', CKPT_DIR.exists())


## §2 — Carica labels/preds per soggetto + cluster

In [ ]:
# Per-trial predictions dal checkpoint EEG_13b
PRED = {}   # sid -> (y_true, y_pred)
BACC_FULL = {}  # sid -> bAcc su tutto il test (sanity)
for p in sorted(CKPT_DIR.glob('P*.pt')):
    sid = int(p.stem[1:])
    ck = torch.load(p, map_location='cpu', weights_only=False)
    if 'labels' not in ck or 'preds' not in ck:
        print(f'P{sid:03d}: NO labels/preds nel checkpoint -> skip'); continue
    y = to_np(ck['labels']).ravel().astype(int)
    yp = to_np(ck['preds']).ravel().astype(int)
    if len(y) != len(yp) or len(y) < 8:
        print(f'P{sid:03d}: shape strane (y={len(y)}, p={len(yp)}) -> skip'); continue
    PRED[sid] = (y, yp)
    BACC_FULL[sid] = balanced_accuracy_score(y, yp)

# Cluster labels EEG_16b
cd = json.loads(CLUSTER_JSON.read_text())
SUBJ_CLUSTER = {int(s): int(l) for s, l in zip(cd['subj_ids'], cd['labels'])}

N_CLASSES = int(max(np.unique(np.concatenate([PRED[s][0] for s in PRED]))) + 1)
CHANCE = 1.0 / N_CLASSES
sids = sorted(PRED)
print(f'Soggetti con predizioni: {len(sids)} | classi: {N_CLASSES} | chance={CHANCE:.3f}')
print(f'test-set size: min={min(len(PRED[s][0]) for s in sids)} '
      f'max={max(len(PRED[s][0]) for s in sids)}')
print(f'bAcc full: mean={np.mean(list(BACC_FULL.values())):.3f} '
      f'range=[{min(BACC_FULL.values()):.3f}, {max(BACC_FULL.values()):.3f}]')
nC0 = sum(SUBJ_CLUSTER.get(s,-1)==0 for s in sids)
nC1 = sum(SUBJ_CLUSTER.get(s,-1)==1 for s in sids)
print(f'C0={nC0}  C1={nC1}  (no-cluster={len(sids)-nC0-nC1})')


## §3 — Split-half reliability

Per ogni ripetizione: split stratificato 50/50 dei trial di test di ogni soggetto → bAcc_A, bAcc_B.
Poi correlazione **tra soggetti** di bAcc_A vs bAcc_B. Media su N_REPS ripetizioni.

Spearman-Brown: lo split-half sottostima la reliability piena → `r_full = 2r / (1 + r)`.

In [ ]:
def split_half_once(subset_sids, seed):
    ba_A, ba_B = [], []
    for sid in subset_sids:
        y, yp = PRED[sid]
        # stratificato: serve >=2 per classe; fallback random se fallisce
        try:
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
            iA, iB = next(sss.split(np.zeros_like(y), y))
        except ValueError:
            idx = np.arange(len(y)); np.random.default_rng(seed).shuffle(idx)
            h = len(idx)//2; iA, iB = idx[:h], idx[h:]
        ba_A.append(balanced_accuracy_score(y[iA], yp[iA]))
        ba_B.append(balanced_accuracy_score(y[iB], yp[iB]))
    return np.array(ba_A), np.array(ba_B)

def reliability(subset_sids, n_reps=N_REPS):
    if len(subset_sids) < 4:
        return None
    rp, rs = [], []
    for b in range(n_reps):
        a, c = split_half_once(subset_sids, SEED0 + b)
        if np.std(a) < 1e-9 or np.std(c) < 1e-9:
            continue
        rp.append(np.corrcoef(a, c)[0, 1])
        rs.append(spearmanr(a, c).statistic)
    rp, rs = np.array(rp), np.array(rs)
    r_mean = float(np.nanmean(rp))
    sb = 2*r_mean / (1 + r_mean) if r_mean > -1 else np.nan
    return {
        'n': len(subset_sids),
        'pearson_mean': r_mean,
        'pearson_ci': (float(np.nanpercentile(rp, 2.5)), float(np.nanpercentile(rp, 97.5))),
        'spearman_mean': float(np.nanmean(rs)),
        'spearman_brown': float(sb),
        'reps': len(rp),
    }

C0 = [s for s in sids if SUBJ_CLUSTER.get(s) == 0]
C1 = [s for s in sids if SUBJ_CLUSTER.get(s) == 1]

REL = {'ALL': reliability(sids), 'C0': reliability(C0), 'C1': reliability(C1)}
for k, v in REL.items():
    if v is None:
        print(f'{k}: n<4, skip'); continue
    print(f'{k:4s} n={v["n"]:2d} | Pearson r={v["pearson_mean"]:+.3f} '
          f'CI[{v["pearson_ci"][0]:+.3f},{v["pearson_ci"][1]:+.3f}] | '
          f'Spearman={v["spearman_mean"]:+.3f} | Spearman-Brown={v["spearman_brown"]:+.3f}')


## §4 — Scatter bAcc_A vs bAcc_B (una ripetizione rappresentativa)

In [ ]:
# split rappresentativo (seed con r più vicino alla media ALL)
target = REL['ALL']['pearson_mean']
best_seed, best_d = SEED0, 1e9
for b in range(N_REPS):
    a, c = split_half_once(sids, SEED0 + b)
    if np.std(a) < 1e-9 or np.std(c) < 1e-9: continue
    d = abs(np.corrcoef(a, c)[0, 1] - target)
    if d < best_d: best_d, best_seed = d, SEED0 + b

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
fig.patch.set_facecolor('white')
COL = {0: '#4A90E2', 1: '#FF8C42'}
for ax, subset, title, rk in [
    (axes[0], sids, 'Tutti i soggetti', 'ALL'),
    (axes[1], C0,   'C0 Fronto-motor',  'C0'),
    (axes[2], C1,   'C1 Fronto-occipital','C1')]:
    a, c = split_half_once(subset, best_seed)
    cols = [COL.get(SUBJ_CLUSTER.get(s), '#888') for s in subset]
    ax.scatter(a, c, c=cols, s=55, edgecolors='#222', lw=0.6, zorder=3)
    lim = [min(a.min(), c.min())-0.02, max(a.max(), c.max())+0.02]
    ax.plot(lim, lim, '--', c='#999', lw=1, zorder=1)
    ax.axhline(CHANCE, c='#ccc', lw=0.8); ax.axvline(CHANCE, c='#ccc', lw=0.8)
    v = REL[rk]
    sub = (f"r={v['pearson_mean']:+.2f}  SB={v['spearman_brown']:+.2f}  n={v['n']}"
           if v else 'n<4')
    ax.set_title(f'{title}\n{sub}', fontsize=11, fontweight='bold')
    ax.set_xlabel('bAcc metà A'); ax.set_ylabel('bAcc metà B')
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')

plt.suptitle('EEG_25 — Split-half reliability della bAcc per soggetto',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
sp = FIG_DIR / 'eeg25_bacc_reliability.png'
plt.savefig(sp, dpi=150, bbox_inches='tight', facecolor='white'); plt.show()
print('Salvato', sp)


## §5 — Verdetto

In [ ]:
def verdict(r_sb):
    if r_sb is None or np.isnan(r_sb): return 'N/A'
    if r_sb >= 0.7: return 'AFFIDABILE - tratto solido, ha senso cercare le feature'
    if r_sb >= 0.5: return 'MODERATO - tratto reale ma rumoroso, predicibilita limitata'
    return 'SCARSO - la bAcc e in gran parte rumore di stima -> domanda MAL POSTA'

print('='*70)
print('EEG_25 - VERDETTO RELIABILITY')
print('='*70)
for k in ['ALL', 'C0', 'C1']:
    v = REL[k]
    if v is None:
        print(f'{k}: n<4'); continue
    sb = v['spearman_brown']
    print(f"\n{k} (n={v['n']}):")
    print(f'  Spearman-Brown reliability = {sb:+.3f}')
    print(f'  -> {verdict(sb)}')
    print(f'  Tetto teorico di varianza spiegabile (R2 max) ~ {max(sb, 0):.2f}')

print('\n' + '='*70)
print('IMPLICAZIONE PER EEG_24/26:')
sbC1 = REL['C1']['spearman_brown'] if REL['C1'] else float('nan')
print(f"  La reliability within-C1 ({sbC1:+.3f}) e il SOFFITTO di EEG_24/26.")
print("  Nessun modello (PCC, PLV, Riemannian, multivariato) puo superarlo.")
print("  Se basso -> il null di EEG_24 e atteso, non un fallimento di metodo.")
print('='*70)


## §6 — Note e limiti

- **Cosa misura**: la reliability del *valore* di bAcc per soggetto (rumore di stima da pochi trial).
- **Cross-session (decay)**: il test set di EEG_13b è un singolo split per soggetto → questa analisi
  NON separa "tratto" da "decadimento cross-sessione". Per quello servono predizioni per-sessione
  (inference su tutte le sessioni). Follow-up se la reliability within-test è alta.
- **Spearman-Brown** corregge il fatto che lo split-half usa metà dei dati: stima la reliability del
  test set pieno.
- **Lettura per la tesi**: qualunque sia l'esito, è un risultato.
  - Alta reliability + EEG_24 null → paradosso forte (tratto reale ma non EEG-predicibile).
  - Bassa reliability → la proficiency è in gran parte rumore → chiude la domanda con onestà.
